# FinGPT

To run this note book the follow dependencies are required:
- transformers==4.38.2
- peft==0.10.0

In [62]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig
from peft import PeftModel
import pandas as pd
from datasets import load_dataset

In [63]:
base_model = "THUDM/chatglm2-6b" 
peft_model = "FinGPT/fingpt-mt_chatglm2-6b_lora" # specialised financial model

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModel.from_pretrained(
    base_model, 
    trust_remote_code=True, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

model = PeftModel.from_pretrained(model, peft_model, token=False)
model = model.eval()

print("Model Loaded!")

/home/jovyan/my_envs/diss_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 7/7 [00:44<00:00,  6.39s/it]


Model Loaded!


In [6]:
def predict_financial_sentiment(instruction, news_text):
    prompt = f"Instruction: {instruction}\nInput: {news_text}\nAnswer: "
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            do_sample=True
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Answer: ")[-1].strip()

instruction = "Here is a news headline. Predict the sentiment as positive, negative or neutral"
news = "Gold prices surged to a record high as investors sought safety amid global market volatility."

result = predict_financial_sentiment(instruction, news)
print(f"Prediction: {result}")

Prediction: positive


## Financial Phrase Bank

In [39]:
def load_fpb_data(file_path):
    data = []
    with open(file_path, "r", encoding="latin-1") as f:
        for line in f:
            # Split only on the first '@' just in case the text contains one
            parts = line.strip().split('@')
            if len(parts) == 2:
                data.append({
                    "headline": parts[0].strip(),
                    "true_label": parts[1].strip()
                })
    return data

fpb_data_all = load_fpb_data("Sentences_AllAgree.txt")
fpb_data_75 = load_fpb_data("Sentences_75Agree.txt")
fpb_data_66 = load_fpb_data("Sentences_66Agree.txt")
fpb_data_50 = load_fpb_data("Sentences_50Agree.txt")

In [40]:
fpb_df_all = pd.DataFrame(fpb_data_all)
fpb_df_75 = pd.DataFrame(fpb_data_75)
fpb_df_66 = pd.DataFrame(fpb_data_66)
fpb_df_50 = pd.DataFrame(fpb_data_50)

In [36]:
def get_sentiment_batch(instruction, headlines, batch_size=64, print_interval=512):
    results = []
    total = len(headlines)
    
    for i in range(0, len(headlines), batch_size):
        batch_texts = headlines[i:i + batch_size]
        prompts = [f"Instruction: {instruction}\nInput: {text}\nAnswer: " for text in batch_texts]
        
        # Tokenize the entire batch
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode batch
        responses = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        for response in responses:
            results.append(response.split("Answer: ")[-1].strip())

        if i > 0 and i % print_interval == 0:
            progress = (i / total) * 100
            print(f"Processed {i} rows ({progress:.1f}% complete)...")
        
    return results

In [37]:
fpb_df_75['sentiment_prediction'] = get_sentiment_batch(instruction, fpb_df_75['headline'].tolist())

Processed 512 rows (14.8% complete)...
Processed 1024 rows (29.7% complete)...
Processed 1536 rows (44.5% complete)...
Processed 2048 rows (59.3% complete)...
Processed 2560 rows (74.1% complete)...
Processed 3072 rows (89.0% complete)...


In [25]:
correct_pred = fpb_df_75[fpb_df_75["true_label"].values == fpb_df_75["sentiment_prediction"].values]
accuracy =  len(correct_pred)/ len(fpb_df_75)
print("---Financial Phrase Bank dataset---\n")
print(f"finGPT accuracy: {accuracy*100:.2f}%")
print("-"*30)

---Financial Phrase Bank dataset---

finGPT accuracy: 86.53%
------------------------------


## FiQa

In [42]:
fiqa_df = pd.read_csv('fiqa_train.csv')

In [43]:
def rough_score_to_label(score):
    if score < -0.1:
        return "negative"
    elif score > 0.1:
        return "positive"
    else:
        return "neutral"

fiqa_df["rough_true_label"] = fiqa_df["score"].apply(rough_score_to_label)

In [44]:
fiqa_df['sentiment_prediction'] = get_sentiment_batch(instruction, fiqa_df['sentence'].tolist())

Processed 512 rows (62.3% complete)...


In [45]:
fiqa_df.head()

,_id,sentence,target,aspect,score,type,rough_true_label,sentiment_prediction
0,1,Royal Mail chairman Donald Brydon set to step ...,Royal Mail,Corporate/Appointment,-0.374,headline,negative,neutral
1,100,Slump in Weir leads FTSE down from record high,Weir,Market/Volatility/Volatility,-0.827,headline,negative,negative
2,1000,AstraZeneca wins FDA approval for key new lung...,AstraZeneca,Corporate/Regulatory,0.549,headline,positive,positive
3,1002,UPDATE 1-Lloyds to cut 945 jobs as part of 3-y...,Lloyds,Corporate/Strategy,-0.266,headline,negative,negative
4,1005,Standard Chartered Shifts Emerging-Markets Str...,Standard Chartered,Corporate/Strategy,-0.461,headline,negative,positive


In [46]:
correct_pred = fiqa_df[fiqa_df["rough_true_label"].values == fiqa_df["sentiment_prediction"].values]
accuracy =  len(correct_pred)/ len(fiqa_df)
print("---FiQa dataset---\n")
print(f"finGPT accuracy: {accuracy*100:.2f}%")
print("-"*30)

---FiQa dataset---

finGPT accuracy: 71.78%
------------------------------


## tfns

In [48]:
def get_tfns():    
    tfns = load_dataset("zeroshot/twitter-financial-news-sentiment")
    df = tfns['train'].to_pandas()
    label_map = {0: "negative", 1: "positive", 2: "neutral"}
    df['label'] = df['label'].map(label_map)
    return df

In [52]:
tfns_df = get_tfns()

In [53]:
tfns_df.head(1)

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,negative


In [54]:
tfns_df['sentiment_prediction'] = get_sentiment_batch(instruction, tfns_df['text'].tolist())

Processed 512 rows (5.4% complete)...
Processed 1024 rows (10.7% complete)...
Processed 1536 rows (16.1% complete)...
Processed 2048 rows (21.5% complete)...
Processed 2560 rows (26.8% complete)...
Processed 3072 rows (32.2% complete)...
Processed 3584 rows (37.6% complete)...
Processed 4096 rows (42.9% complete)...
Processed 4608 rows (48.3% complete)...
Processed 5120 rows (53.7% complete)...
Processed 5632 rows (59.0% complete)...
Processed 6144 rows (64.4% complete)...
Processed 6656 rows (69.7% complete)...
Processed 7168 rows (75.1% complete)...
Processed 7680 rows (80.5% complete)...
Processed 8192 rows (85.8% complete)...
Processed 8704 rows (91.2% complete)...
Processed 9216 rows (96.6% complete)...


In [57]:
correct_pred = tfns_df[tfns_df["label"].values == tfns_df["sentiment_prediction"].values]
accuracy =  len(correct_pred)/ len(tfns_df)
print("---Twitter Financial News dataset---\n")
print(f"finBERT accuracy: {accuracy*100:.2f}%")
print("-"*30)

---Twitter Financial News dataset---

finBERT accuracy: 72.44%
------------------------------
